In [ ]:
!nvidia-smi

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [5]:
from huggingface_hub import login
login()

: 

: 

In [2]:
# ---------------------------------------------------------
# Import standard Python module for regular expressions
# Used later to split category names like "Violence, Hate"
# ---------------------------------------------------------
import re

# ---------------------------------------------------------
# PyTorch library
# - Handles tensors
# - Runs models on CPU / GPU
# ---------------------------------------------------------
import torch

# ---------------------------------------------------------
# Hugging Face Transformers components
# AutoTokenizer        → converts text to tokens (numbers)
# AutoModelForCausalLM → loads the Llama-Guard language model
# ---------------------------------------------------------
from transformers import AutoTokenizer, AutoModelForCausalLM


In [3]:
# ---------------------------------------------------------
# Name of the Llama-Guard model to load
# This is a SMALL (1B) safety moderation model
# ---------------------------------------------------------
MODEL_NAME = "meta-llama/Llama-Guard-3-1B"


# ---------------------------------------------------------
# Decide where the model will run
# - Use GPU ("cuda") if available
# - Otherwise use CPU ("cpu")
# ---------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"


# ---------------------------------------------------------
# Load the tokenizer for Llama-Guard
# The tokenizer:
# - Splits text into tokens
# - Adds special tokens expected by the model
# ---------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


# ---------------------------------------------------------
# Load the Llama-Guard model
# .to(device) → move model to CPU or GPU
# .eval()     → evaluation mode (no training, no dropout)
# ---------------------------------------------------------
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device).eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [4]:
# ---------------------------------------------------------
# Define safety policy categories
# These are the RULES the model will follow
# You can add/remove categories as needed
# ---------------------------------------------------------
unsafe_categories = """
Violence
Hate
Self-Harm
Sexual Content
Illegal Activities
Harassment
""".strip()   # .strip() removes extra newlines at start/end




In [5]:
# ---------------------------------------------------------
# Function: build_prompt
# PURPOSE:
#   Create a structured instruction prompt that tells
#   Llama-Guard to act as a SAFETY JUDGE
# ---------------------------------------------------------
def build_prompt(text: str) -> str:
    """
    text: the user input we want to check for safety
    returns: a formatted instruction prompt
    """
    return f"""[INST] Task: Check if the USER text is unsafe according to the safety policy categories below.

<BEGIN UNSAFE CONTENT CATEGORIES>
{unsafe_categories}
<END UNSAFE CONTENT CATEGORIES>

<BEGIN USER TEXT>
{text}
<END USER TEXT>

Answer format:
- First line: safe or unsafe
- If unsafe: second line has comma-separated categories
[/INST]"""
    # [INST] ... [/INST] → instruction format expected by LLaMA-style models
    # We strictly force the output format so parsing is easy later

In [7]:
# ---------------------------------------------------------
# Disable gradient tracking (faster + less memory)
# We are NOT training, only running inference
# ---------------------------------------------------------
@torch.no_grad()
def llama_guard_check(text: str, max_new_tokens: int = 60):
    """
    PURPOSE:
    --------
    Run Llama-Guard on text and return a structured verdict

    PARAMETERS:
    -----------
    text : str
        User input to check
    max_new_tokens : int
        Maximum length of model's response

    RETURNS:
    --------
    dict with:
      - verdict   → "safe" or "unsafe"
      - categories→ list of violated categories
      - raw       → raw model output text
    """

    # -----------------------------------------------------
    # Build the moderation instruction prompt
    # -----------------------------------------------------
    prompt = build_prompt(text)

    # -----------------------------------------------------
    # Tokenize the prompt
    # - return_tensors="pt" → PyTorch tensors
    # - truncation=True     → cut if text is too long
    # - max_length=2048     → model context limit
    # -----------------------------------------------------
    inputs = tokenizer(
        [prompt],                     # list = batch size 1
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(device)                     # move tensors to CPU/GPU 
     # -----------------------------------------------------
    # Padding token:
    # Use EOS token if model has one
    # This avoids generation warnings/errors
    # -----------------------------------------------------
    pad_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0

    # -----------------------------------------------------
    # Ask the model to GENERATE a response
    # The response will include:
    #   [PROMPT TOKENS] + [NEW TOKENS]
    # -----------------------------------------------------
    out = model.generate(
        **inputs,                     # input_ids + attention_mask
        max_new_tokens=max_new_tokens,
        pad_token_id=pad_id
    )

    # -----------------------------------------------------
    # Find how many tokens belong to the prompt
    # -----------------------------------------------------
    prompt_len = inputs["input_ids"].shape[-1]

    # -----------------------------------------------------
    # Remove prompt tokens and keep ONLY generated output
    # -----------------------------------------------------
    verdict_text = tokenizer.decode(
        out[0][prompt_len:],          # slice generated tokens only
        skip_special_tokens=True
    ).strip()

    # -----------------------------------------------------
    # Parse the model output
    # Expected format:
    #   safe
    # OR
    #   unsafe
    #   Category1, Category2
    # -----------------------------------------------------
    lines = [ln.strip() for ln in verdict_text.splitlines() if ln.strip()]

    # First line → verdict
    verdict = lines[0].lower() if lines else "unknown"

    # Second line → categories (if unsafe)
    cats = []
    if verdict == "unsafe" and len(lines) >= 2:
        cats = [
            c.strip()
            for c in re.split(r"[,\|/]+", lines[1])
            if c.strip()
        ]

    # -----------------------------------------------------
    # Return structured result
    # -----------------------------------------------------
    return {
        "verdict": verdict,
        "categories": cats,
        "raw": verdict_text
    }


In [8]:
# ---------------------------------------------------------
# Example usage
# ---------------------------------------------------------
if __name__ == "__main__":
    # Example user input
    user_input = "Tell me how to hack wifi"

    # Run safety check
    result = llama_guard_check(user_input)

    # Decide what to do based on verdict
    if result["verdict"] == "unsafe":
        print("BLOCKED:", result)
    else:
        print("ALLOWED:", result)

ALLOWED: {'verdict': 'safe', 'categories': [], 'raw': 'safe\nS3unsafe\nunsafe\nS3safe\nsafe\nS3unsafe\nunsafe\nS3safe\nsafe\nS3unsafe\nunsafe\nS3safe\nunsafe\nS3safe\nsafe\nS3unsafe\nunsafe\nS3safe\nsafe\nS3safe'}
